In [6]:
from entsoe import EntsoePandasClient
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
client = EntsoePandasClient(api_key="be9993ee-346c-45b9-bef1-201dd8f8bf3d")

In [9]:
load_dotenv("/Users/raynapatel/Downloads/Zeus-Energy_Security_Index/datasets/entsoe/entsoe.env")

True

In [11]:
os.makedirs("data/raw", exist_ok=True)
prices.to_csv("data/raw/day_ahead_prices_DE.csv")
print("Pulled successfully. Shape:", prices.shape)
print(prices.head())

Pulled successfully. Shape: (35037,)
2021-01-01 00:00:00+01:00    50.87
2021-01-01 01:00:00+01:00    48.19
2021-01-01 02:00:00+01:00    44.68
2021-01-01 03:00:00+01:00    42.92
2021-01-01 04:00:00+01:00    40.39
dtype: float64


In [15]:
from entsoe import EntsoePandasClient
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()
client = EntsoePandasClient(api_key=os.getenv("ENTSOE_API_KEY"))

start = pd.Timestamp("2021-01-01", tz="Europe/Berlin")
end   = pd.Timestamp("2024-12-31", tz="Europe/Berlin")

prices = client.query_day_ahead_prices("DE_LU", start=start, end=end)

# Save raw output
prices.to_csv("data/raw/day_ahead_prices_DE.csv")
print("Pulled successfully. Shape:", prices.shape)
print(prices.head(10))

Pulled successfully. Shape: (35037,)
2021-01-01 00:00:00+01:00    50.87
2021-01-01 01:00:00+01:00    48.19
2021-01-01 02:00:00+01:00    44.68
2021-01-01 03:00:00+01:00    42.92
2021-01-01 04:00:00+01:00    40.39
2021-01-01 05:00:00+01:00    40.20
2021-01-01 06:00:00+01:00    39.63
2021-01-01 07:00:00+01:00    40.09
2021-01-01 08:00:00+01:00    41.27
2021-01-01 09:00:00+01:00    44.88
dtype: float64


In [19]:
import pandas as pd
import numpy as np
import os

# --- Load ---
df = pd.read_csv("data/raw/day_ahead_prices_DE.csv")
df.columns = ["timestamp", "price_eur_mwh"]

# --- Parse timestamps (API format is already ISO8601) ---
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert("Europe/Berlin")
df = df.sort_values("timestamp").reset_index(drop=True)

# --- Check for missing values ---
print(f"Missing values: {df['price_eur_mwh'].isna().sum()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# --- Fill small gaps (forward fill up to 3 hours) ---
df["price_eur_mwh"] = df["price_eur_mwh"].ffill(limit=3)

# --- Add time features ---
df["date"]       = df["timestamp"].dt.date
df["hour"]       = df["timestamp"].dt.hour
df["dayofweek"]  = df["timestamp"].dt.dayofweek
df["month"]      = df["timestamp"].dt.month
df["year"]       = df["timestamp"].dt.year
df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

# --- Daily average ---
daily = df.groupby("date")["price_eur_mwh"].mean().reset_index()
daily.columns = ["date", "avg_price_eur_mwh"]
daily["date"] = pd.to_datetime(daily["date"])

# --- Save ---
os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/prices_hourly_DE.csv", index=False)
daily.to_csv("data/clean/prices_daily_DE.csv", index=False)

print("Saved. Shape:", daily.shape)
print(daily.head(35))

Missing values: 0
Date range: 2021-01-01 00:00:00+01:00 to 2024-12-30 23:00:00+01:00
Saved. Shape: (1460, 2)
         date  avg_price_eur_mwh
0  2021-01-01          48.398333
1  2021-01-02          50.562500
2  2021-01-03          38.622500
3  2021-01-04          48.017917
4  2021-01-05          55.337083
5  2021-01-06          52.682917
6  2021-01-07          70.927083
7  2021-01-08          78.336250
8  2021-01-09          59.318750
9  2021-01-10          52.164583
10 2021-01-11          48.152500
11 2021-01-12          46.100417
12 2021-01-13          42.472083
13 2021-01-14          70.577083
14 2021-01-15          72.891250
15 2021-01-16          55.845417
16 2021-01-17          53.998750
17 2021-01-18          55.953750
18 2021-01-19          44.954167
19 2021-01-20          36.282083
20 2021-01-21          30.074583
21 2021-01-22          42.529167
22 2021-01-23          51.720833
23 2021-01-24          49.167500
24 2021-01-25          60.837083
25 2021-01-26          61.373750
